#  Optimizing a Model with LLM Compressor

In this notebook, we'll:
1. See how **post-training quantization** works via full precision & compressed model comparisons
2. Use the `llm-compressor` library to apply a GPTQ recipe that produces a W4A16 quantized model
3. Test and evaluate the quantized model against the original

## What is LLM Compressor?

[llm-compressor](https://github.com/vllm-project/llm-compressor) is the production quantization toolkit from the vLLM project. It takes a trained model and reduces precision in a single pass, no retraining required.

The core API is **`oneshot`**: you give it a model, a calibration dataset (small sample of real inputs used to minimize quantization error), and a recipe describing how to quantize (e.g. GPTQ, W4A16). It produces a smaller model that can be served directly by [vLLM](https://github.com/vllm-project/vllm), an LLM inference engine that you'll use in the next lesson.

```python
oneshot(
    model="model-name",           # HuggingFace model ID
    dataset="dataset-name",       # Calibration dataset
    recipe=recipe,                # Quantization configuration
    output_dir="./outputs",        # Where to save
    num_calibration_samples=256,  # Samples for calibration
    max_seq_length=4096,          # Sequence length
)
```

The name "oneshot" reflects that this happens in a **single pass** over calibration data, no retraining required.

## Setup

In [ ]:
%pip install torch transformers llmcompressor datasets

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os, gc, math, pathlib
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

os.environ["TOKENIZERS_PARALLELISM"] = "false"

MODEL_DIR = "./models/Qwen3-0.6B"
OUTPUT_DIR = "./outputs/Qwen3-0.6B-W4A16"

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")

model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

print(f"Base Model: {MODEL_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")

Base Model: ./models/Qwen3-0.6B
Output Directory: ./outputs/Qwen3-0.6B-W4A16


## Define the Recipe

A **recipe** tells LLM Compressor how to quantize. It's a list of modifiers: each one specifies an algorithm and settings we'll apply to the model.

### Picking the Quantization Algorithm

| Name | Type | Description |
|------|------|-------------|
| `Round-to-Nearest (RTN)` | Weights & Activations | **Fastest compression** method with baseline accuracy. Best for **CPU/Mac** deployments (GGUF) and **experimentation**. |
| `AWQ` | Weights Only | Best **accuracy–speed tradeoff** for **NVIDIA** GPUs. Faster calibration than GPTQ and highly preferred for **INT4** quantization. |
| `GPTQ` | Weights Only | Mature and **widely supported**. Requires substantial **VRAM** during quantization but remains an industry standard for **compatibility**. |
| `SparseGPT` | Sparsity | Enables **extreme compression** through **sparsification**. Most effective on hardware with sparse acceleration such as **NVIDIA H100** (2:4 sparsity). |
| `SmoothQuant` | Transforms & Smoothing | One of the best methods for** INT8 weight and activation** quantization. **Reduces activation outliers** to improve quantization quality. |
| `SpinQuant` | Transforms & Smoothing | Uses **learned rotations** to improve quantization quality and can quantize models **without** requiring a **calibration dataset**. |
| `QuIP` | Transforms & Smoothing | **Advanced low-bit quantization** approach that can operate **without a calibration dataset**, making it suitable for **low-compute** environments. |

### Available Modifiers

**Quantization Modifiers:**

| Modifier | Description |
|:--|:--|
| `GPTQModifier` | GPTQ algorithm: uses calibration data to find optimal quantization values |
| `AWQModifier` | Activation-Weighted Quantization preserves salient weights (the weights that matter most) |
| `AutoRoundModifier` | Intel's algorithm with learnable rounding/clipping |
| `QuantizationModifier` | Basic PTQ and QAT for simple use cases |

**Transform Modifiers** (for improving accuracy):

| Modifier | Description |
|:--|:--|
| `SmoothQuantModifier` | Smooths activations before quantization, often paired with GPTQ |
| `QuIPModifier` | Hadamard rotations to reduce outliers |
| `SpinQuantModifier` | SpinQuant-style rotations to even out weight distributions |

**Note:** Modifiers can be chained: e.g. applying `SmoothQuantModifier` before `GPTQModifier` improves accuracy for W8A8 quantization.

### Quantization Schemes

The `scheme` parameter determines the bit-width for weights (W) and activations (A):

| Scheme | Weights | Activations | Quantized Layer Reduction | Quality Impact |
|:--|:--|:--|:--|:--|
| `W8A16` | 8-bit | 16-bit (FP16) | ~50% | Minimal |
| `W4A16` | 4-bit | 16-bit (FP16) | ~75% | Low–Moderate |
| `W8A8` | 8-bit | 8-bit | ~50% | Low |
| `W4A8` | 4-bit | 8-bit | ~75% | Moderate |


**Note:** These reductions apply to the quantized layers only. The embedding and `lm_head` layers are kept at full precision, so total model size reduction depends on how large those layers are relative to the rest. For small models (~0.6B), expect ~40–50% total reduction with W4A16.

### Our Recipe: GPTQModifier with W4A16

| Parameter | Value | Why |
|:--|:--|:--|
| `scheme` | `W4A16` | 4-bit weights  |
| `targets` | `Linear` | Linear layers hold most parameters - biggest savings |
| `ignore` | `["lm_head"]` | Output layer maps to vocabulary - keep it precise |

In [10]:
from llmcompressor.modifiers.quantization import GPTQModifier

recipe = GPTQModifier(
    scheme="W4A16",
    targets="Linear",
    ignore=["lm_head"]
)

print(f"Recipe: {recipe}")

Recipe: config_groups=None targets=['Linear'] ignore=['lm_head'] scheme='W4A16' kv_cache_scheme=None weight_observer=None input_observer=None output_observer=None observer=None bypass_divisibility_checks=False index=None group=None start=None end=None update=None initialized_=False finalized_=False started_=False ended_=False block_size=128 dampening_frac=0.01 actorder=static offload_hessians=False


## Quantize the Model

### Why a calibration dataset?

GPTQ doesn't just round weights to lower precision, it uses a small set of real text to measure how each weight affects the model's output, then finds quantized values that minimize the error. This is what makes it more accurate than naive rounding.

The `dataset` parameter specifies what text to use for calibration. Here you'll use [WikiText-2](https://huggingface.co/datasets/mindchain/wikitext2), a standard benchmark of Wikipedia articles, the same dataset you'll use later for perplexity evaluation. You only need a few hundred samples as calibration is fast.

>**Note:** Since quantization can take several minutes and benefits from a GPU, we've already run it ahead of time and provided the quantized model in the `Qwen3-0.6B-W4A16` folder (`OUTPUT_DIR`). This learning environment is memory-constrained, so it might crash if you run the quantization yourself. The `if not os.path.isdir(OUTPUT_DIR)` check below ensures you skip re-running quantization when the folder already exists, so you can move straight to evaluation.

In [ ]:
from llmcompressor import oneshot

if not os.path.isdir(OUTPUT_DIR):
    oneshot(
        model="Qwen/Qwen3-0.6B",
        dataset="wikitext",
        dataset_config_name="wikitext-2-raw-v1",
        recipe=recipe,
        output_dir=OUTPUT_DIR,
        max_seq_length=1024,    # 4096
        num_calibration_samples=32,  # 256
    )
    print(f"Quantization complete. Model saved to: {OUTPUT_DIR}")

## Compare Model Sizes

Let's see how much space quantization saves.

In [20]:
def folder_size(path):
    p = pathlib.Path(path)
    if not p.exists():
        return 0
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file())

def format_size(nbytes):
    if nbytes < 1024**2:
        return f"{nbytes/1024:.1f} KB"
    if nbytes < 1024**3:
        return f"{nbytes/1024**2:.1f} MB"
    return f"{nbytes/1024**3:.2f} GB"

size_orig = folder_size(MODEL_DIR)
size_q = folder_size(OUTPUT_DIR)
reduction = (1 - size_q / size_orig) * 100 if size_orig > 0 else 0

print("Model Size Comparison")
print("=" * 45)
print(f"Original (BF16):    {format_size(size_orig)}")
print(f"Quantized (W4A16):  {format_size(size_q)}")
print(f"Reduction:          {reduction:.0f}%")

Model Size Comparison
Original (BF16):    1.12 GB
Quantized (W4A16):  524.4 MB
Reduction:          54%


> **Note**:
>
> We might expect a 75% reduction since we went from 16-bit to 4-bit weights (4x smaller), but the actual reduction is 54%. The reason: only the **linear layer weights** are quantized to Int4. The rest of the model (including the LM head and normalization layers) stays at higher precision.
>
> So the 4x compression applies to the bulk of the parameters (the linear layers, which dominate the model), but the unquantized pieces pull the overall reduction down to ~54%. This ratio improves with larger models, where linear weights make up an even bigger share of total size — a 70B model quantized the same way gets much closer to the theoretical 4x.

## Test Both Models

Smaller files are only useful if the model still produces reasonable output. Let's generate text from both and compare using the Hugging Face [Transformers](https://huggingface.co/docs/transformers/en/index) library, starting with the original model **then** the quantized model.

In [24]:
prompt = "Machine learning is a branch of"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR, device_map="cpu", dtype=torch.bfloat16,
)

inputs = tokenizer(prompt, return_tensors="pt")
outputs = base_model.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)
generated = outputs[0][inputs["input_ids"].shape[-1]:]

print(f"Base Model ({MODEL_DIR})")
print(f"Prompt: {prompt}")
print(f"Response: {tokenizer.decode(generated, skip_special_tokens=True)}")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Base Model (/content/drive/MyDrive/models/Qwen3-0.6B)
Prompt: Machine learning is a branch of
Response:  artificial intelligence that has gained significant attention in recent years, particularly in the context of the rise of big data and the need for efficient, scalable solutions to complex problems. As the field continues to evolve, the integration of machine learning into various industries is becoming increasingly widespread. However, despite its growing popularity,


In [25]:
import logging
logging.getLogger("llmcompressor").setLevel(logging.WARNING)

quant_model = AutoModelForCausalLM.from_pretrained(
    OUTPUT_DIR, device_map="cpu", dtype=torch.bfloat16,
)

inputs = tokenizer(prompt, return_tensors="pt")
outputs = quant_model.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)
generated = outputs[0][inputs["input_ids"].shape[-1]:]

print(f"Quantized Model ({OUTPUT_DIR})")
print(f"Prompt: {prompt}")
print(f"Response: {tokenizer.decode(generated, skip_special_tokens=True)}")

Compressing model: 100%|██████████| 196/196 [00:01<00:00, 186.62it/s]


Loading weights:   0%|          | 0/702 [00:00<?, ?it/s]

Decompressing model: 100%|██████████| 196/196 [00:03<00:00, 53.27it/s]


Quantized Model (/content/drive/MyDrive/outputs/Qwen3-0.6B-W4A16)
Prompt: Machine learning is a branch of
Response:  artificial intelligence that has been gaining popularity in recent years. It is a field that is expected to become more and more important in the future. However, there are some challenges that are still present in the field. For example, the problem of data scarcity and the problem of data quality. These two problems


## Perplexity Comparison

Side-by-side text gives intuition, but **perplexity** is the standard metric: it measures how well the model predicts text. Lower is better. If quantization has degraded the model, its perplexity will be noticeably higher.

In [27]:
from datasets import load_dataset

def calculate_perplexity(model, tokenizer, dataset, max_tokens=5000, stride=512):
    encodings = tokenizer(
        "\n\n".join(dataset["text"]),
        return_tensors="pt", truncation=True, max_length=max_tokens,
    )
    input_ids = encodings.input_ids
    nlls, prev_end = [], 0

    for begin_loc in range(0, input_ids.size(1), stride):
        end_loc = min(begin_loc + stride, input_ids.size(1))
        trg_len = end_loc - prev_end
        input_slice = input_ids[:, begin_loc:end_loc]
        target_slice = input_slice.clone()
        target_slice[:, :-trg_len] = -100
        with torch.no_grad():
            loss = model(input_slice, labels=target_slice).loss
            nlls.append(loss * trg_len)
        prev_end = end_loc

    return math.exp(torch.stack(nlls).sum() / prev_end)

from datasets import load_dataset

test_data = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
    split="test"
)
print(f"Loaded {len(test_data)} test samples")

Loaded 4358 test samples


In [28]:
quant_ppl = calculate_perplexity(quant_model, tokenizer, test_data)
print(f"Quantized perplexity: {quant_ppl:.2f}")

Quantized perplexity: 38.34


In [29]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR, device_map="cpu", dtype=torch.bfloat16,
)
base_ppl = calculate_perplexity(base_model, tokenizer, test_data)
print(f"Base perplexity: {base_ppl:.2f}")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Base perplexity: 32.78


In [30]:
print("Perplexity Comparison")
print("=" * 40)
print(f"Base (BF16):      {base_ppl:.2f}")
print(f"Quantized (W4A16): {quant_ppl:.2f}")
print(f"Difference:       {quant_ppl - base_ppl:+.2f} ({(quant_ppl/base_ppl - 1)*100:+.1f}%)")
print(f"\nA small increase in perplexity is expected — the quantized layers use 4-bit weights.")

Perplexity Comparison
Base (BF16):      32.78
Quantized (W4A16): 38.34
Difference:       +5.56 (+17.0%)

A small increase in perplexity is expected — the quantized layers use 4-bit weights.


> **Note:**
>
> A 17% increase in perplexity suggests noticeable but acceptable degradation for a highly compressed 4-bit model. The degradation may be influenced by the limited calibration setup used during quantization to fit within Colab memory constraints. Further improvements may be possible by increasing the number of calibration samples or using alternative quantization methods such as AWQ.

## Summary

In this notebook, we:

- Learned how the LLM Compressor **`oneshot`** applies post-training quantization with a GPTQ recipe
- Compared model sizes: W4A16 reduces weights from 16-bit to 4-bit
- Tested both models on the same prompt to verify output quality
- Measured **perplexity** to quantify the accuracy tradeoff

## Resources

- [LLM Compressor GitHub](https://github.com/vllm-project/llm-compressor)
- [LLM Compressor Docs](https://docs.vllm.ai/projects/llm-compressor/en/latest/)